# Huấn luyện CNN–LSTM tuần tự theo từng dataset

Notebook train một model tuần tự độc lập cho từng dataset `kaggle`, `csic` và `obfuscation`. Mỗi model chỉ fit tokenizer trên train split của chính dataset đó; các split được tạo trực tiếp bởi `preprocessing/preprocess_data.py`. Artifact được lưu riêng dưới `cnn_lstm/artifacts_cnn_lstm_rerun/by_dataset/<dataset>/`.


## 1. Thiết lập môi trường và cấu hình


In [1]:
from pathlib import Path
import argparse
import importlib.util
import json
import os
import random
import sys

START_DIR = Path.cwd().resolve()
if (START_DIR / "cnn_lstm" / "CNN_LSTM.py").exists():
    PROJECT_ROOT = START_DIR
    NOTEBOOK_DIR = PROJECT_ROOT / "cnn_lstm"
elif START_DIR.name == "cnn_lstm" and (START_DIR / "CNN_LSTM.py").exists():
    NOTEBOOK_DIR = START_DIR
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError("Hãy chạy notebook từ repo root hoặc thư mục cnn_lstm.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(NOTEBOOK_DIR)

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display

from preprocessing import preprocess_data as prep

module_path = NOTEBOOK_DIR / "CNN_LSTM.py"
spec = importlib.util.spec_from_file_location("sequential_cnn_lstm_pipeline", module_path)
pipeline = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pipeline
spec.loader.exec_module(pipeline)

SEED = 42
MAX_LEN = 1024
EMBEDDING_DIM = 64
BATCH_SIZE = 128
EPOCHS = 50
TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN_VAL = 0.10
DECISION_THRESHOLD = 0.50
EARLY_STOPPING_MIN_DELTA = 1e-4

KAGGLE_PATH = PROJECT_ROOT / "SQLInjection_XSS_MixDataset.1.0.0.csv"
CSIC_PATH = PROJECT_ROOT / "csic_database.csv"
OBFUSCATION_PATH = PROJECT_ROOT / "obfuscation_dataset.xlsx"
ARTIFACT_ROOT = NOTEBOOK_DIR / "artifacts_cnn_lstm_rerun"
PARALLEL_ARTIFACT_ROOT = PROJECT_ROOT / "cnn_lstm_parallel" / "artifacts"

for required_path in (
    module_path,
    PROJECT_ROOT / "preprocessing" / "preprocess_data.py",
    KAGGLE_PATH,
    CSIC_PATH,
    OBFUSCATION_PATH,
):
    if not required_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file bắt buộc: {required_path}")

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ARGS = argparse.Namespace(
    max_len=MAX_LEN,
    embedding_dim=EMBEDDING_DIM,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED,
)

print("Project root:", PROJECT_ROOT)
print("Preprocessing module:", Path(prep.__file__).resolve())
print("Sequential artifact root:", ARTIFACT_ROOT)
print("Parallel artifact root:  ", PARALLEL_ARTIFACT_ROOT)
print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))


Project root: C:\Users\admin\Desktop\obfuscated-web-attack-detection
Preprocessing module: C:\Users\admin\Desktop\obfuscated-web-attack-detection\preprocessing\preprocess_data.py
Sequential artifact root: C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm\artifacts_cnn_lstm_rerun
Parallel artifact root:   C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm_parallel\artifacts
TensorFlow: 2.21.0
GPU available: False


## 2. Tiền xử lý và chia riêng từng dataset

Notebook không viết lại loader, clean hoặc split. Toàn bộ dữ liệu được tạo bằng `prep.build_dataset_splits(...)` từ `preprocessing/preprocess_data.py`:

- `kaggle` và `csic`: stratified row split.
- `obfuscation`: group split theo `original_pattern` khi cột này tồn tại.
- Tỷ lệ toàn bộ dữ liệu: 72% train, 8% validation, 20% test.
- Không URL decode, không HTML unescape, không lowercase; chỉ chuẩn hóa khoảng trắng.


In [2]:
dataset_splits, preprocessing_metadata = prep.build_dataset_splits(
    kaggle_path=str(KAGGLE_PATH),
    csic_path=str(CSIC_PATH),
    obfu_path=str(OBFUSCATION_PATH),
    test_size=TEST_SIZE,
    val_size=VAL_SIZE_WITHIN_TRAIN_VAL,
    seed=SEED,
)

processed_data_dir = ARTIFACT_ROOT / "processed_data_by_dataset"
prep.save_dataset_splits(dataset_splits, processed_data_dir)
with (ARTIFACT_ROOT / "preprocessing_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(preprocessing_metadata, file, ensure_ascii=False, indent=2)

audit_rows = []
for dataset_name, splits in dataset_splits.items():
    clean_summary = preprocessing_metadata["datasets"][dataset_name]["clean"]
    for split_name, frame in splits.items():
        label_counts = frame["label"].value_counts().to_dict()
        audit_rows.append({
            "dataset": dataset_name,
            "split": split_name,
            "rows": int(len(frame)),
            "normal": int(label_counts.get(0, 0)),
            "attack": int(label_counts.get(1, 0)),
            "attack_rate": float(frame["label"].mean()),
            "clean_dataset_rows": int(clean_summary["rows"]),
        })

split_audit = pd.DataFrame(audit_rows)
display(split_audit.style.format({"attack_rate": "{:.2%}"}))
print("Đã lưu split tại:", processed_data_dir)


,dataset,split,rows,normal,attack,attack_rate,clean_dataset_rows
0,kaggle,train,109197,38647,70550,64.61%,151663
1,kaggle,val,12133,4294,7839,64.61%,151663
2,kaggle,test,30333,10736,19597,64.61%,151663
3,csic,train,8248,3348,4900,59.41%,11457
4,csic,val,917,372,545,59.43%,11457
5,csic,test,2292,930,1362,59.42%,11457
6,obfuscation,train,217559,107998,109561,50.36%,300000
7,obfuscation,val,21982,12001,9981,45.41%,300000
8,obfuscation,test,60459,30001,30458,50.38%,300000


Đã lưu split tại: C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm\artifacts_cnn_lstm_rerun\processed_data_by_dataset


## 3. Kiến trúc CNN–LSTM tuần tự

Luồng đặc trưng: `Embedding → Conv1D(k=3) → MaxPool → Conv1D(k=5) → MaxPool → LSTM(128, return_sequences=True) → GlobalMaxPooling1D → Dense → Sigmoid`.

Các cấu hình dùng để so sánh với song song được đồng bộ: `MAX_LEN=1024`, embedding 64, hai CNN 128 filters, LSTM 128 units, Dense 64, dropout 0.3, padding/truncating `post`, threshold 0.5 và EarlyStopping theo `val_loss` với `min_delta=1e-4` và patience 3.


In [3]:
demo_vocab_size = max(
    len(pipeline.build_tokenizer(splits["train"]["payload"]).word_index) + 1
    for splits in dataset_splits.values()
)
demo_model = pipeline.build_model(
    vocab_size=demo_vocab_size,
    max_len=MAX_LEN,
    embedding_dim=EMBEDDING_DIM,
)
demo_model.summary()
print("Parameter count (demo vocab):", demo_model.count_params())
del demo_model
tf.keras.backend.clear_session()


Model: "Hybrid_1D_CNN_LSTM_Sequence_Pooling_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │        13,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3 (Conv1D)                │ (None, 1024, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5 (Conv1D)                │ (None, 256, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_context_sequence (LSTM)    │ (None, 64, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_global_max_pool            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 260,225 (1016.50 KB)

 Trainable params: 260,225 (1016.50 KB)

 Non-trainable params: 0 (0.00 B)

Parameter count (demo vocab): 260225



## 4. Train và test riêng từng dataset

Mỗi phần tử trong `DATASETS_TO_TRAIN` tạo một model và tokenizer riêng. Kết quả cùng miền để báo cáo nằm tại `evaluation[dataset_name]`; script cũng lưu cross-dataset evaluation để phân tích khả năng chuyển miền, nhưng không dùng các test khác để học hoặc chọn model.

Có thể chỉ chạy một dataset bằng cách sửa danh sách, ví dụ `DATASETS_TO_TRAIN = ["csic"]`.


In [ ]:
DATASETS_TO_TRAIN = ["kaggle", "csic", "obfuscation"]

unknown_datasets = set(DATASETS_TO_TRAIN) - set(dataset_splits)
if unknown_datasets:
    raise ValueError(f"Dataset không hợp lệ: {sorted(unknown_datasets)}")

all_results = {}
cross_eval_rows = []

for dataset_name in DATASETS_TO_TRAIN:
    model_result, model_rows = pipeline.train_and_evaluate_source_model(
        train_source=dataset_name,
        dataset_splits=dataset_splits,
        output_dir=ARTIFACT_ROOT,
        args=EXPERIMENT_ARGS,
    )
    all_results[dataset_name] = model_result
    cross_eval_rows.extend(model_rows)

    # Ghi tổng hợp sau từng dataset để các kết quả đã hoàn thành không bị mất
    # nếu dataset tiếp theo bị interrupt.
    cross_eval_results = pd.DataFrame(cross_eval_rows)
    cross_eval_results.to_csv(
        ARTIFACT_ROOT / "cross_eval_results.csv",
        index=False,
        encoding="utf-8",
    )
    if not cross_eval_results.empty:
        cross_eval_results.pivot(
            index="train_source",
            columns="test_source",
            values="accuracy",
        ).to_csv(
            ARTIFACT_ROOT / "cross_eval_accuracy_matrix.csv",
            encoding="utf-8",
        )

    experiment_results = {
        "dataset_metadata": preprocessing_metadata,
        "trained_sources": list(all_results),
        "models": all_results,
        "artifacts": {
            "processed_data": str(processed_data_dir),
            "models": str(ARTIFACT_ROOT / "by_dataset"),
            "cross_eval_results": str(ARTIFACT_ROOT / "cross_eval_results.csv"),
            "cross_eval_accuracy_matrix": str(ARTIFACT_ROOT / "cross_eval_accuracy_matrix.csv"),
        },
    }
    with (ARTIFACT_ROOT / "cross_eval_results.json").open("w", encoding="utf-8") as file:
        json.dump(experiment_results, file, ensure_ascii=False, indent=2)

print("Đã train xong:", ", ".join(all_results))
print("Artifact theo dataset:", ARTIFACT_ROOT / "by_dataset")



=== TRAINING MODEL FOR KAGGLE ===
Train: 109,197 | Val: 12,133
Vocabulary size: 212
X_train: (109197, 1024) | X_val: (12133, 1024)
Class weights: {0: 1.41274872564494, 1: 0.7738979447200567}
Epoch 1/50
609/854 ━━━━━━━━━━━━━━━━━━━━ 1:25 350ms/step - accuracy: 0.9190 - auc: 0.9745 - loss: 0.1560 - precision: 0.9738 - recall: 0.8934

## 5. Thống kê kết quả CNN–LSTM tuần tự theo từng dataset

Bảng thống kê dùng đúng test split của dataset đã train model. Ngoài metric tổng quát, bảng giữ riêng Precision, Recall, F1 của lớp Attack, False Positive và False Negative.


In [ ]:
split_statistics_rows = []
dataset_statistics_rows = []
trained_results = globals().get("all_results", {}).copy()

for dataset_name in dataset_splits:
    if dataset_name not in trained_results:
        result_path = (
            ARTIFACT_ROOT
            / "by_dataset"
            / dataset_name
            / "metadata_and_results.json"
        )
        if result_path.exists():
            with result_path.open("r", encoding="utf-8") as file:
                trained_results[dataset_name] = json.load(file)

    for split_name, frame in dataset_splits[dataset_name].items():
        lengths = frame["payload"].str.len()
        label_counts = frame["label"].value_counts().to_dict()
        split_statistics_rows.append({
            "dataset": dataset_name,
            "split": split_name,
            "rows": int(len(frame)),
            "normal": int(label_counts.get(0, 0)),
            "attack": int(label_counts.get(1, 0)),
            "attack_rate": float(frame["label"].mean()),
            "mean_length": float(lengths.mean()),
            "p95_length": float(lengths.quantile(0.95)),
            "max_length": int(lengths.max()),
            "over_max_len": int((lengths > MAX_LEN).sum()),
            "over_max_len_rate": float((lengths > MAX_LEN).mean()),
        })

    row = {
        "dataset": dataset_name,
        "train_rows": int(len(dataset_splits[dataset_name]["train"])),
        "validation_rows": int(len(dataset_splits[dataset_name]["val"])),
        "test_rows": int(len(dataset_splits[dataset_name]["test"])),
        "test_accuracy": None,
        "test_auc_roc": None,
        "attack_precision": None,
        "attack_recall": None,
        "attack_f1": None,
        "false_positives": None,
        "false_negatives": None,
    }
    if dataset_name in trained_results:
        test_result = trained_results[dataset_name]["evaluation"][dataset_name]
        attack_report = test_result["classification_report"]["Attack (1)"]
        matrix = test_result["confusion_matrix"]
        row.update({
            "test_accuracy": float(test_result["accuracy"]),
            "test_auc_roc": test_result["auc_roc"],
            "attack_precision": float(attack_report["precision"]),
            "attack_recall": float(attack_report["recall"]),
            "attack_f1": float(attack_report["f1-score"]),
            "false_positives": int(matrix[0][1]),
            "false_negatives": int(matrix[1][0]),
        })
    dataset_statistics_rows.append(row)

dataset_statistics = pd.DataFrame(dataset_statistics_rows)
split_statistics = pd.DataFrame(split_statistics_rows)

dataset_statistics.to_csv(
    ARTIFACT_ROOT / "dataset_statistics.csv",
    index=False,
    encoding="utf-8",
)
split_statistics.to_csv(
    ARTIFACT_ROOT / "dataset_split_statistics.csv",
    index=False,
    encoding="utf-8",
)

display(dataset_statistics.style.format({
    "test_accuracy": "{:.4f}",
    "test_auc_roc": "{:.4f}",
    "attack_precision": "{:.4f}",
    "attack_recall": "{:.4f}",
    "attack_f1": "{:.4f}",
}, na_rep="chưa train"))
display(split_statistics.style.format({
    "attack_rate": "{:.2%}",
    "mean_length": "{:.1f}",
    "p95_length": "{:.1f}",
    "over_max_len_rate": "{:.2%}",
}))


Các kết luận kiến trúc chỉ nên dựa trên những dataset có đủ hai artifact tuần tự và song song, cùng split, seed, tokenizer rule, padding, ngưỡng 0.5 và checkpoint tốt nhất theo `val_loss`. Cell cuối tự kiểm tra các điều kiện này trước khi đưa dataset vào bảng và biểu đồ.


In [ ]:
# 6. So sánh CNN–LSTM tuần tự với CNN ∥ LSTM song song
import matplotlib.pyplot as plt
from IPython.display import Markdown

display(Markdown("## 6. So sánh CNN–LSTM tuần tự với CNN ∥ LSTM song song"))

COMPARE_DATASETS = ["kaggle", "csic", "obfuscation"]
METRIC_NAMES = ["Precision", "Recall", "F1-score"]
ARCHITECTURE_NAMES = ["CNN–LSTM tuần tự", "CNN ∥ LSTM song song"]
COLORS = ["#5B9BD5", "#ED7D31", "#A5A5A5"]


def load_json(path):
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def contract_value(payload, key):
    preprocessing = payload.get("preprocessing", {})
    if key in preprocessing:
        return preprocessing[key]
    return payload.get("model", {}).get(key)


def validate_comparison_contract(dataset_name, sequential_payload, parallel_payload):
    if sequential_payload.get("train_source") != dataset_name:
        raise ValueError(
            f"Sequential artifact không khớp {dataset_name}: "
            f"train_source={sequential_payload.get('train_source')!r}"
        )
    if parallel_payload.get("dataset") != dataset_name:
        raise ValueError(
            f"Parallel artifact không khớp {dataset_name}: "
            f"dataset={parallel_payload.get('dataset')!r}"
        )

    checks = {
        "seed": (contract_value(sequential_payload, "seed"), contract_value(parallel_payload, "seed")),
        "max_len": (contract_value(sequential_payload, "max_len"), contract_value(parallel_payload, "max_len")),
        "padding": (contract_value(sequential_payload, "padding"), contract_value(parallel_payload, "padding")),
        "truncating": (contract_value(sequential_payload, "truncating"), contract_value(parallel_payload, "truncating")),
        "decision_threshold": (
            contract_value(sequential_payload, "decision_threshold"),
            contract_value(parallel_payload, "decision_threshold"),
        ),
        "url_decode": (contract_value(sequential_payload, "url_decode"), contract_value(parallel_payload, "url_decode")),
        "html_unescape": (contract_value(sequential_payload, "html_unescape"), contract_value(parallel_payload, "html_unescape")),
        "lowercase": (contract_value(sequential_payload, "lowercase"), contract_value(parallel_payload, "lowercase")),
        "tokenizer_fit_split": (
            contract_value(sequential_payload, "tokenizer_fit_split"),
            contract_value(parallel_payload, "tokenizer_fit_split"),
        ),
        "group_column": (contract_value(sequential_payload, "group_column"), contract_value(parallel_payload, "group_column")),
        "embedding_dim": (contract_value(sequential_payload, "embedding_dim"), contract_value(parallel_payload, "embedding_dim")),
        "cnn_filters": (contract_value(sequential_payload, "cnn_filters"), contract_value(parallel_payload, "cnn_filters")),
        "cnn_kernel_sizes": (
            contract_value(sequential_payload, "cnn_kernel_sizes"),
            contract_value(parallel_payload, "cnn_kernel_sizes"),
        ),
        "cnn_pool_sizes": (
            contract_value(sequential_payload, "cnn_pool_sizes"),
            contract_value(parallel_payload, "cnn_pool_sizes"),
        ),
        "lstm_units": (contract_value(sequential_payload, "lstm_units"), contract_value(parallel_payload, "lstm_units")),
        "lstm_return_sequences": (
            contract_value(sequential_payload, "lstm_return_sequences"),
            contract_value(parallel_payload, "lstm_return_sequences"),
        ),
        "lstm_pooling": (contract_value(sequential_payload, "lstm_pooling"), contract_value(parallel_payload, "lstm_pooling")),
        "dense_units": (contract_value(sequential_payload, "dense_units"), contract_value(parallel_payload, "dense_units")),
        "dropout": (contract_value(sequential_payload, "dropout"), contract_value(parallel_payload, "dropout")),
        "learning_rate": (contract_value(sequential_payload, "learning_rate"), contract_value(parallel_payload, "learning_rate")),
        "early_stopping_monitor": (
            contract_value(sequential_payload, "early_stopping_monitor"),
            contract_value(parallel_payload, "early_stopping_monitor"),
        ),
        "early_stopping_patience": (
            contract_value(sequential_payload, "early_stopping_patience"),
            contract_value(parallel_payload, "early_stopping_patience"),
        ),
        "early_stopping_min_delta": (
            contract_value(sequential_payload, "early_stopping_min_delta"),
            contract_value(parallel_payload, "early_stopping_min_delta"),
        ),
        "vocab_size": (
            contract_value(sequential_payload, "vocab_size"),
            contract_value(parallel_payload, "vocab_size"),
        ),
    }

    sequential_splits = sequential_payload.get("preprocessing", {}).get("splits", {})
    parallel_splits = parallel_payload.get("preprocessing", {}).get("splits", {})
    for split_name in ("train", "val", "test"):
        checks[f"{split_name}_rows"] = (
            sequential_splits.get(split_name, {}).get("rows"),
            parallel_splits.get(split_name, {}).get("rows"),
        )

    mismatches = {
        key: values
        for key, values in checks.items()
        if values[0] != values[1]
        or (key != "group_column" and (values[0] is None or values[1] is None))
    }
    if mismatches:
        details = ", ".join(
            f"{key}: sequential={values[0]!r}, parallel={values[1]!r}"
            for key, values in mismatches.items()
        )
        raise ValueError(f"Hợp đồng so sánh {dataset_name} chưa khớp: {details}")
    return checks


comparison_rows = []
comparison_metrics = {}
comparison_contracts = {}
missing_pairs = []

for dataset_name in COMPARE_DATASETS:
    sequential_path = (
        ARTIFACT_ROOT
        / "by_dataset"
        / dataset_name
        / "metadata_and_results.json"
    )
    parallel_path = (
        PARALLEL_ARTIFACT_ROOT
        / dataset_name
        / "metadata_and_results.json"
    )
    missing = [path for path in (sequential_path, parallel_path) if not path.exists()]
    if missing:
        missing_pairs.append((dataset_name, missing))
        continue

    sequential_payload = load_json(sequential_path)
    parallel_payload = load_json(parallel_path)
    comparison_contracts[dataset_name] = validate_comparison_contract(
        dataset_name,
        sequential_payload,
        parallel_payload,
    )

    sequential_test = sequential_payload["evaluation"][dataset_name]
    parallel_test = parallel_payload["evaluation"]["test"]
    comparison_metrics[dataset_name] = []

    for architecture, payload, test_result in (
        (ARCHITECTURE_NAMES[0], sequential_payload, sequential_test),
        (ARCHITECTURE_NAMES[1], parallel_payload, parallel_test),
    ):
        attack = test_result["classification_report"]["Attack (1)"]
        values = [
            100.0 * float(attack["precision"]),
            100.0 * float(attack["recall"]),
            100.0 * float(attack["f1-score"]),
        ]
        comparison_metrics[dataset_name].append(values)
        comparison_rows.append({
            "Dataset": dataset_name.upper(),
            "Architecture": architecture,
            "Parameters": int(payload["model"]["parameter_count"]),
            "Accuracy": 100.0 * float(test_result["accuracy"]),
            "AUC-ROC": 100.0 * float(test_result["auc_roc"]),
            "Attack Precision": values[0],
            "Attack Recall": values[1],
            "Attack F1": values[2],
        })

if missing_pairs:
    print("Cảnh báo: bỏ qua các dataset chưa đủ hai artifact:")
    for dataset_name, paths in missing_pairs:
        print(f"- {dataset_name.upper()}:")
        for path in paths:
            print("   ", path)

if not comparison_rows:
    raise FileNotFoundError(
        "Chưa có dataset nào đủ artifact tuần tự và song song để so sánh. "
        "Hãy train ít nhất một model tuần tự bằng cell train phía trên."
    )

comparison_table = pd.DataFrame(comparison_rows)
comparison_table.to_csv(
    ARTIFACT_ROOT / "sequential_vs_parallel_by_dataset.csv",
    index=False,
    encoding="utf-8",
)
display(comparison_table.style.format({
    "Accuracy": "{:.2f}%",
    "AUC-ROC": "{:.2f}%",
    "Attack Precision": "{:.2f}%",
    "Attack Recall": "{:.2f}%",
    "Attack F1": "{:.2f}%",
}))

plotted_datasets = list(comparison_metrics)
all_values = np.asarray([
    values
    for dataset_values in comparison_metrics.values()
    for values in dataset_values
])
y_min = max(0.0, np.floor((float(all_values.min()) - 5.0) / 5.0) * 5.0)

fig, axes = plt.subplots(
    1,
    len(plotted_datasets),
    figsize=(6 * len(plotted_datasets), 5.5),
    squeeze=False,
    sharey=True,
)
for axis, dataset_name in zip(axes[0], plotted_datasets):
    values = np.asarray(comparison_metrics[dataset_name])
    x = np.arange(len(ARCHITECTURE_NAMES))
    width = 0.22
    for metric_index, (metric_name, color) in enumerate(zip(METRIC_NAMES, COLORS)):
        bars = axis.bar(
            x + (metric_index - 1) * width,
            values[:, metric_index],
            width,
            label=metric_name.upper(),
            color=color,
        )
        axis.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)

    axis.set_title(dataset_name.upper(), fontsize=14)
    axis.set_xticks(x)
    axis.set_xticklabels(["Tuần tự", "Song song"])
    axis.set_ylim(y_min, 100.5)
    axis.grid(axis="y", alpha=0.30)
    axis.set_axisbelow(True)
    for spine in ("top", "right"):
        axis.spines[spine].set_visible(False)

axes[0][0].set_ylabel("Giá trị (%)")
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.suptitle("CNN–LSTM TUẦN TỰ VS CNN ∥ LSTM SONG SONG", fontsize=16)
fig.tight_layout(rect=(0, 0.08, 1, 0.94))

comparison_chart_path = ARTIFACT_ROOT / "sequential_vs_parallel_by_dataset.png"
fig.savefig(comparison_chart_path, dpi=300, bbox_inches="tight")
plt.show()

print("Đã kiểm tra hợp đồng công bằng cho:", ", ".join(name.upper() for name in plotted_datasets))
print("Đã lưu bảng:", ARTIFACT_ROOT / "sequential_vs_parallel_by_dataset.csv")
print("Đã lưu biểu đồ:", comparison_chart_path)
